# S&P 500 横截面量价信号验证


1. `Signal existence`：纯量价特征能不能预测未来 10 天截面残差收益
2. `Model comparison`：LightGBM 是否优于简单基线
3. `Portfolio translation`：信号翻译成组合后，问题出在信号还是组合构建
4. `Regime explanation`：regime 先作为解释层，而不是先验交易规则

当前版本的边界：
- 股票池使用当前 S&P 500 成分股，存在 survivorship bias
- 标签仍使用 10 天 beta-adjusted residual return
- 本 notebook 首先服务于研究诊断，不等价于最终可交易策略


In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "equity").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from equity.data import load_universe, load_spy
from equity.features import compute_all_features, cross_sectional_rank
from equity.labels import make_binary_up_labels, make_labels
from equity.backtest import backtest_topN, compute_daily_returns
from equity.analysis import (
    calc_ic_series,
    quantile_analysis,
    summary_metrics,
    plot_equity_curve,
    summarize_oos_windows,
    plot_oos_window_equity_curves,
)

OUTPUT_DIR = REPO_ROOT / "output"
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")


## 0. 研究前提、核心假设与实验定义

### 研究前提

- 仅凭量价特征，直接预测未来 `n` 天收益幅度很难，尤其在股票横截面上噪音很大。
- 与其一开始追求“预测涨多少”，更现实的目标是先判断“未来 `n` 天上涨的概率谁更高”。
- 如果模型能稳定选出上涨概率最高的 `top N` 股票，那么它有机会转化成更好的收益/回撤特征，但这件事需要单独验证，不能直接假定成立。
- 当前股票池使用当前 S&P 500 成分股，因此存在 survivorship bias；这属于当前版本接受的近似边界。

### 核心假设

- H1：纯量价特征可以提供未来 `n` 天上涨概率的 OOS 截面信息。 OOS: out-of-sample
- H2：分类任务比回归未来收益幅度更容易学习，更适合作为量价特征的第一阶段目标。
- H3：预测上涨概率最高的 `top N` 股票，会呈现更高的胜率，并有可能带来更平滑的权益曲线。
- H4：高胜率不自动等于高收益，因此必须同时检查平均盈利、平均亏损、交易成本后收益与回撤。

### 当前主方向与对照方向

| 角色 | formulation | 标签定义 | 用途 |
|---|---|---|---|
| 主方向 | Classification | `future_n_day_raw_return > 0` | 直接回答“谁更可能上涨” |
| 对照方向 | Regression | `future_n_day_beta_adjusted_residual_return` | 作为旧方案基线，比较信息损失与排序效果 |

这里的关键改变是：
- 主任务不再默认定义为“收益幅度预测”
- 主任务改为“上涨概率排序”
- 回归版本保留，但更多作为 baseline 和对照实验

### 分类版实验设计

1. 标签：未来 `n` 天 raw return 是否大于 0。
2. 模型输出：每只股票未来 `n` 天上涨的预测概率。
3. 选股：每个调仓日买入预测上涨概率最高的 `top N` 股票。
4. 对照：保留回归版与简单 baseline，避免只因为分类更容易训练就误判它更好。

### 主评估指标

| 层级 | 核心问题 | 主要指标 |
|---|---|---|
| Signal existence | 有没有稳定的上涨概率信息 | Top-N hit rate, daily cross-sectional hit rate |
| Portfolio translation | 高胜率能否转成真实收益 | Annual return, Sharpe, max drawdown, turnover |
| Risk quality | 赢的时候赚多少，输的时候亏多少 | Avg win, avg loss, payoff ratio |
| Relative performance | 是否真的优于大盘 | Excess return vs SPY, beta, down-capture |

只有当“高胜率”能够在收益、回撤和相对表现上同时站住，这个方向才算真正成立。


In [8]:
# notebook 内研究版训练流程的参数配置。
CONFIG = {
    # 股票数据缓存目录。
    "cache_dir": REPO_ROOT / "data" / "equity",
    # 预测未来多少个交易日。
    "label_horizon": 10,
    # 计算 residual baseline 时使用的滚动 beta 窗口。
    "beta_window": 252,
    # walk-forward 切分的折数。
    "n_splits": 5,
    # 每一折训练集长度，单位是交易日。
    "train_days": 750,
    # 验证集长度，只用于 early stopping。
    "val_days": 60,
    # 最终样本外测试集长度。
    "test_days": 60,
    # train/val 和 val/test 之间的间隔，用来避免标签重叠泄漏。
    "gap_days": 10,
    # boosting 最大轮数上限。
    "num_boost_round": 300,
    # 验证集指标连续这么多轮不提升就停止训练。
    "early_stopping_rounds": 50,
    # 组合回测中假设的单边交易成本，单位 bps。
    "cost_bps": 10.0,
    # 主策略视角下实际买入的 top N 股票数量。
    "live_top_n": 10,
    # 诊断和稳健性分析时使用的更宽松 top N。
    "diag_top_n": 30,
    # 把预测概率转成 0/1 时使用的分类阈值。
    "probability_threshold": 0.5,
}

CONFIG


{'cache_dir': PosixPath('/home/renjie/dev/Invest/data/equity'),
 'label_horizon': 10,
 'beta_window': 252,
 'n_splits': 5,
 'train_days': 750,
 'val_days': 60,
 'test_days': 60,
 'gap_days': 10,
 'num_boost_round': 300,
 'early_stopping_rounds': 50,
 'cost_bps': 10.0,
 'live_top_n': 10,
 'diag_top_n': 30,
 'probability_threshold': 0.5}

## 1. 数据加载

这里只做最少的数据准备：
- 股票面板数据
- SPY 基准
- raw daily return，用于真实 PnL 回测

如果这里的数据边界没有定义清楚，后面的任何结论都不稳。


In [9]:
panel = load_universe(CONFIG["cache_dir"])
spy = load_spy(CONFIG["cache_dir"])
daily_returns = compute_daily_returns(panel)

print(f"Panel rows: {len(panel):,}")
print(f"Tickers: {panel.index.get_level_values('ticker').nunique()}")
print(f"Date range: {panel.index.get_level_values('date').min().date()} -> {panel.index.get_level_values('date').max().date()}")
panel.head()


Panel rows: 1,375,458
Tickers: 502
Date range: 2015-01-02 -> 2026-03-31


Price                   open       high        low      close       volume
date       ticker                                                         
2015-01-02 A       37.535488  37.653983  36.797173  36.970360    1529200.0
           AAPL    24.671145  24.682220  23.776348  24.214888  212818400.0
           ABBV    41.470293  42.078659  41.470293  41.755463    5086100.0
           ABT     36.517570  36.678974  36.025289  36.235115    3216600.0
           ACGL    18.764398  18.884845  18.472788  18.539352    1101600.0

## 2. 特征与标签

这一层只回答一件事：我们到底在拿什么去预测什么。

- 特征：25 个纯量价横截面特征
- 主标签：未来 10 天 raw return 是否大于 0
- 对照标签：未来 10 天 beta-adjusted residual return
- 变换：特征做一次 cross-sectional rank，分类标签保留 `0/1`


In [10]:
raw_features = compute_all_features(panel)
features = cross_sectional_rank(raw_features)

# Main label: whether the stock goes up over the next N trading days.
labels = make_binary_up_labels(
    panel,
    periods=CONFIG["label_horizon"],
)

# Keep the old regression target as a baseline / comparison track.
residual_labels = make_labels(
    panel,
    spy,
    periods=CONFIG["label_horizon"],
    beta_window=CONFIG["beta_window"],
)

print(f"Feature shape: {features.shape}")
print(f"Binary label count: {labels.notna().sum():,}")
print(f"Positive rate: {labels.mean():.2%}")
features.head()


Feature shape: (1375458, 25)
Binary label count: 1,370,438
Positive rate: 55.86%


ret_5d  ret_10d  ret_20d  ret_60d  ret_120d  skip5_ret_20d  \
date       ticker                                                               
2015-01-02 A          NaN      NaN      NaN      NaN       NaN            NaN   
           AAPL       NaN      NaN      NaN      NaN       NaN            NaN   
           ABBV       NaN      NaN      NaN      NaN       NaN            NaN   
           ABT        NaN      NaN      NaN      NaN       NaN            NaN   
           ACGL       NaN      NaN      NaN      NaN       NaN            NaN   

                   skip5_ret_60d  skip20_ret_120d  efficiency_20d  \
date       ticker                                                   
2015-01-02 A                 NaN              NaN             NaN   
           AAPL              NaN              NaN             NaN   
           ABBV              NaN              NaN             NaN   
           ABT               NaN              NaN             NaN   
           ACGL              NaN              NaN             NaN   

                   efficiency_60d  up_ratio_20d  ret_concentration_20d  \
date       ticker                                                        
2015-01-02 A                  NaN           NaN                    NaN   
           AAPL               NaN           NaN                    NaN   
           ABBV               NaN           NaN                    NaN   
           ABT                NaN           NaN                    NaN   
           ACGL               NaN           NaN                    NaN   

                   max_dd_20d  dist_high_60d  ret_vol_20d  ret_vol_60d  \
date       ticker                                                        
2015-01-02 A              NaN            NaN          NaN          NaN   
           AAPL           NaN            NaN          NaN          NaN   
           ABBV           NaN            NaN          NaN          NaN   
           ABT            NaN            NaN          NaN          NaN   
           ACGL           NaN            NaN          NaN          NaN   

                   slope_rmse_20d  slope_rmse_60d  abnormal_vol_5d  \
date       ticker                                                    
2015-01-02 A                  NaN             NaN              NaN   
           AAPL               NaN             NaN              NaN   
           ABBV               NaN             NaN              NaN   
           ABT                NaN             NaN              NaN   
           ACGL               NaN             NaN              NaN   

                   abnormal_vol_20d  vwap_ret_20d  vwap_ret_60d  \
date       ticker                                                 
2015-01-02 A                    NaN           NaN           NaN   
           AAPL                 NaN           NaN           NaN   
           ABBV                 NaN           NaN           NaN   
           ABT                  NaN           NaN           NaN   
           ACGL                 NaN           NaN           NaN   

                   up_vol_ratio_20d  reversal_vol_5d  vol_price_corr_20d  
date       ticker                                                         
2015-01-02 A                    NaN              NaN                 NaN  
           AAPL                 NaN              NaN                 NaN  
           ABBV                 NaN              NaN                 NaN  
           ABT                  NaN              NaN                 NaN  
           ACGL                 NaN              NaN                 NaN

## 3. 主实验：LightGBM OOS 分类预测


新的时间结构是：
- `train -> gap -> val -> gap -> test`
- `val` 只用于 early stopping
- `test` 完全不参与轮数选择和调参



In [11]:
# notebook 实验里使用的 LightGBM 超参数。
NOTEBOOK_LGBM_PARAMS = {
    # 二分类任务：预测未来 N 日收益是否为正的概率。
    "objective": "binary",
    # 用于 early stopping 的验证指标。
    "metric": "binary_logloss",
    # 每棵树最多多少个叶子；越大越灵活，也越容易过拟合。
    "num_leaves": 31,
    # 树的最大深度，限制模型复杂度。
    "max_depth": 6,
    # 每一轮 boosting 的学习率；越小通常越稳，但需要更多轮。
    "learning_rate": 0.05,
    # 每棵树随机采样的特征比例。
    "feature_fraction": 0.7,
    # 每次 bagging 随机采样的样本比例。
    "bagging_fraction": 0.7,
    # bagging 的频率；5 表示每 5 轮做一次。
    "bagging_freq": 5,
    # L1 正则，抑制过于尖锐的树结构。
    "lambda_l1": 0.1,
    # L2 正则，让模型更平滑。
    "lambda_l2": 0.1,
    # 每个叶子最少样本数；越大表示分裂越保守。
    "min_child_samples": 20,
    # 关闭 LightGBM 每轮训练日志。
    "verbose": -1,
    # 先固定用 CPU，保证 notebook 环境里结果更稳定。
    "device_type": "cpu",
    # 随机种子，方便结果复现。
    "seed": 42,
}


def binary_logloss(y_true: pd.Series | np.ndarray, y_prob: pd.Series | np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    y_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return float(-(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob)).mean())


def prepare_aligned_panel(
    features: pd.DataFrame,
    labels: pd.Series,
    raw_labels: pd.Series | None = None,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    common_idx = features.index.intersection(labels.index)
    if raw_labels is not None:
        common_idx = common_idx.intersection(raw_labels.index)

    X = features.loc[common_idx]
    y = labels.loc[common_idx].astype(float)
    actual = raw_labels.loc[common_idx].astype(float) if raw_labels is not None else y.copy()

    valid_mask = X.notna().all(axis=1) & y.notna() & actual.notna()
    X = X.loc[valid_mask]
    y = y.loc[valid_mask]
    actual = actual.loc[valid_mask]
    return X, y, actual


def make_walk_forward_splits(
    dates: pd.DatetimeIndex,
    n_splits: int,
    train_days: int,
    val_days: int,
    test_days: int,
    gap_days: int,
) -> list[dict]:
    unique_dates = dates.unique().sort_values()
    n_dates = len(unique_dates)

    total_per_fold = train_days + gap_days + val_days + gap_days + test_days
    if total_per_fold > n_dates:
        raise ValueError(
            f"Not enough dates ({n_dates}) for one fold; need {total_per_fold} days."
        )

    available_for_splits = n_dates - total_per_fold
    step = max(1, available_for_splits // max(1, n_splits - 1)) if n_splits > 1 else 0

    splits = []
    for i in range(n_splits):
        train_start = i * step
        train_end = train_start + train_days
        val_start = train_end + gap_days
        val_end = val_start + val_days
        test_start = val_end + gap_days
        test_end = test_start + test_days

        if test_end > n_dates:
            break

        splits.append(
            {
                "fold": i + 1,
                "train_dates": unique_dates[train_start:train_end],
                "val_dates": unique_dates[val_start:val_end],
                "test_dates": unique_dates[test_start:test_end],
            }
        )

    return splits


def run_notebook_binary_cv(
    aligned_features: pd.DataFrame,
    aligned_labels: pd.Series,
    aligned_actual: pd.Series,
    splits: list[dict],
    params: dict,
    num_boost_round: int,
    early_stopping_rounds: int,
) -> tuple[pd.DataFrame, list, pd.DataFrame]:
    dates = aligned_features.index.get_level_values("date")
    predictions_by_fold = []
    fold_rows = []
    models = []

    for split in splits:
        train_mask = dates.isin(split["train_dates"])
        val_mask = dates.isin(split["val_dates"])
        test_mask = dates.isin(split["test_dates"])

        X_train = aligned_features.loc[train_mask]
        y_train = aligned_labels.loc[train_mask]
        X_val = aligned_features.loc[val_mask]
        y_val = aligned_labels.loc[val_mask]
        X_test = aligned_features.loc[test_mask]
        y_test = aligned_labels.loc[test_mask]

        print(
            f"Fold {split['fold']}/{len(splits)} | "
            f"train {split['train_dates'][0].date()}→{split['train_dates'][-1].date()} ({len(X_train):,}) | "
            f"val {split['val_dates'][0].date()}→{split['val_dates'][-1].date()} ({len(X_val):,}) | "
            f"test {split['test_dates'][0].date()}→{split['test_dates'][-1].date()} ({len(X_test):,})"
        )

        train_set = lgb.Dataset(X_train, label=y_train)
        val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
        callbacks = [
            lgb.early_stopping(early_stopping_rounds),
            lgb.log_evaluation(0),
        ]

        model = lgb.train(
            params,
            train_set,
            num_boost_round=num_boost_round,
            valid_sets=[val_set],
            valid_names=["val"],
            callbacks=callbacks,
        )
        models.append(model)

        best_iteration = model.best_iteration or num_boost_round
        val_pred = model.predict(X_val, num_iteration=best_iteration)
        test_pred = model.predict(X_test, num_iteration=best_iteration)
        baseline_prob = float(y_train.mean())
        baseline_pred = np.full(len(y_test), baseline_prob)

        fold_rows.append(
            {
                "fold": split["fold"],
                "train_start": split["train_dates"][0].date(),
                "train_end": split["train_dates"][-1].date(),
                "val_start": split["val_dates"][0].date(),
                "val_end": split["val_dates"][-1].date(),
                "test_start": split["test_dates"][0].date(),
                "test_end": split["test_dates"][-1].date(),
                "train_positive_rate": baseline_prob,
                "val_positive_rate": float(y_val.mean()),
                "test_positive_rate": float(y_test.mean()),
                "best_iteration": int(best_iteration),
                "val_logloss": binary_logloss(y_val, val_pred),
                "test_logloss": binary_logloss(y_test, test_pred),
                "baseline_test_logloss": binary_logloss(y_test, baseline_pred),
                "test_brier": float(((test_pred - y_test.values) ** 2).mean()),
            }
        )

        fold_pred = pd.DataFrame(
            {
                "prediction": test_pred,
                "actual": aligned_actual.loc[test_mask].values,
                "fold": split["fold"],
            },
            index=X_test.index,
        )
        predictions_by_fold.append(fold_pred)

    predictions = pd.concat(predictions_by_fold).sort_index()
    fold_metrics = pd.DataFrame(fold_rows)
    return predictions, models, fold_metrics


aligned_features, aligned_labels, aligned_actual = prepare_aligned_panel(
    features,
    labels,
    raw_labels=labels,
)

cv_splits = make_walk_forward_splits(
    aligned_features.index.get_level_values("date"),
    n_splits=CONFIG["n_splits"],
    train_days=CONFIG["train_days"],
    val_days=CONFIG["val_days"],
    test_days=CONFIG["test_days"],
    gap_days=CONFIG["gap_days"],
)

predictions, models, fold_metrics = run_notebook_binary_cv(
    aligned_features,
    aligned_labels,
    aligned_actual,
    splits=cv_splits,
    params=NOTEBOOK_LGBM_PARAMS,
    num_boost_round=CONFIG["num_boost_round"],
    early_stopping_rounds=CONFIG["early_stopping_rounds"],
)

display(fold_metrics)
predictions.head()


Fold 1/5 | train 2015-06-25→2018-06-15 (351,776) | val 2018-07-02→2018-09-25 (28,620) | test 2018-10-10→2019-01-07 (28,615)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[42]	val's binary_logloss: 0.67723
Fold 2/5 | train 2017-04-10→2020-04-01 (358,216) | val 2020-04-17→2020-07-13 (29,160) | test 2020-07-28→2020-10-20 (29,220)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[35]	val's binary_logloss: 0.665597
Fold 3/5 | train 2019-01-25→2022-01-13 (364,510) | val 2022-01-31→2022-04-26 (29,700) | test 2022-05-11→2022-08-05 (29,718)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	val's binary_logloss: 0.722922
Fold 4/5 | train 2020-11-06→2023-10-31 (370,366) | val 2023-11-15→2024-02-12 (29,880) | test 2024-02-28→2024-05-22 (29,920)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	val's bina

,fold,train_start,train_end,val_start,val_end,test_start,test_end,train_positive_rate,val_positive_rate,test_positive_rate,best_iteration,val_logloss,test_logloss,baseline_test_logloss,test_brier
0,1,2015-06-25,2018-06-15,2018-07-02,2018-09-25,2018-10-10,2019-01-07,0.571773,0.586373,0.493482,42,0.677230,0.708287,0.705442,0.257459
1,2,2017-04-10,2020-04-01,2020-04-17,2020-07-13,2020-07-28,2020-10-20,0.575351,0.621399,0.536037,35,0.665597,0.692536,0.693689,0.249676
2,3,2019-01-25,2022-01-13,2022-01-31,2022-04-26,2022-05-11,2022-08-05,0.582999,0.452963,0.624975,1,0.722922,0.665037,0.665240,0.236045
3,4,2020-11-06,2023-10-31,2023-11-15,2024-02-12,2024-02-28,2024-05-22,0.536804,0.644177,0.512066,1,0.674663,0.693967,0.694084,0.250408
4,5,2022-08-24,2025-08-20,2025-09-05,2025-11-28,2025-12-15,2026-03-12,0.551307,0.517215,0.501959,25,0.694211,0.696730,0.698036,0.251781


prediction  actual  fold
date       ticker                          
2018-10-10 A         0.560776     0.0     1
           AAPL      0.586143     0.0     1
           ABBV      0.580753     0.0     1
           ABT       0.602448     0.0     1
           ACGL      0.602200     0.0     1

## 4. 第一层：Signal Existence

这里只有信号有效性，不讨论 long-only SPY 增强。

分类版这里看的是：
- 预测概率最高的 `top N` 股票，真实上涨率是否显著高于总体基准
- 概率分桶后的上涨率是否单调
- `0.5` 阈值下的整体命中率是否合理
- 各个 OOS window 是否极度依赖个别窗口


In [ ]:
def daily_top_n_hit_rate(pred_df: pd.DataFrame, top_n: int) -> pd.Series:
    def _calc(group: pd.DataFrame) -> float:
        n = min(top_n, len(group))
        if n == 0:
            return np.nan
        return group.nlargest(n, "prediction")["actual"].mean()

    out = pred_df.groupby(level="date").apply(_calc)
    out.name = f"top_{top_n}_hit_rate"
    return out


def daily_bottom_n_hit_rate(pred_df: pd.DataFrame, top_n: int) -> pd.Series:
    def _calc(group: pd.DataFrame) -> float:
        n = min(top_n, len(group))
        if n == 0:
            return np.nan
        return group.nsmallest(n, "prediction")["actual"].mean()

    out = pred_df.groupby(level="date").apply(_calc)
    out.name = f"bottom_{top_n}_hit_rate"
    return out


top_hit = daily_top_n_hit_rate(predictions, CONFIG["live_top_n"])
bottom_hit = daily_bottom_n_hit_rate(predictions, CONFIG["live_top_n"])
quantile_positive_rate = quantile_analysis(predictions, n_groups=5)
predicted_class = (predictions["prediction"] >= CONFIG["probability_threshold"]).astype(float)

signal_summary = pd.Series({
    "base_positive_rate": predictions["actual"].mean(),
    "overall_hit_rate@0.5": (predicted_class == predictions["actual"]).mean(),
    "brier_score": ((predictions["prediction"] - predictions["actual"]) ** 2).mean(),
    f"top_{CONFIG['live_top_n']}_hit_rate": top_hit.mean(),
    f"bottom_{CONFIG['live_top_n']}_hit_rate": bottom_hit.mean(),
    "top_minus_bottom_hit_rate": top_hit.mean() - bottom_hit.mean(),
})

display(signal_summary.to_frame("value"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
top_hit.plot(ax=axes[0], title=f"Daily Top-{CONFIG['live_top_n']} Hit Rate")
axes[0].axhline(predictions["actual"].mean(), color="black", linewidth=1, linestyle="--")

quantile_positive_rate.mean().plot(kind="bar", ax=axes[1], title="Positive Rate by Probability Quantile")
axes[1].axhline(predictions["actual"].mean(), color="black", linewidth=1, linestyle="--")
plt.tight_layout()


## 5. 第二层：Model Comparison

这一层必须回答：LightGBM 是真的有增量，还是只是一个复杂但不必要的模型。

建议固定 3 个 baseline：
- 单特征 baseline：`ret_20d` 或 `skip5_ret_60d`
- 简单 composite：几类动量特征等权平均
- 线性 baseline：Ridge / ElasticNet

只有当 LightGBM 在 OOS `top-N hit rate`、`Brier score` 和 `top-bottom spread` 上持续优于这些 baseline，复杂模型才算站住。


In [ ]:
# TODO:
# 1. 用相同的 train/test splits 跑 baseline predictions
# 2. 统一输出 top-N hit rate / brier score / top-bottom spread
# 3. 明确记录 LightGBM 是否稳定优于 baseline

baseline_results = pd.DataFrame(
    columns=["model", "top_n_hit_rate", "brier_score", "top_bottom_spread", "notes"]
)
baseline_results


## 6. 第三层：Portfolio Translation

这里要把两个组合问题严格分开：

- `诊断组合`：验证排序有没有赚钱能力，尽量降低 beta / sector 暴露干扰
- `目标组合`：long-only active vs SPY，评估真实产品形态

如果 long-only 组合跑输，不能立刻说信号没用；要先看诊断组合到底有没有把 alpha 变成 spread。


In [ ]:
bt_results = backtest_topN(
    predictions,
    daily_returns,
    top_n=CONFIG["live_top_n"],
    rebalance_days=CONFIG["label_horizon"],
    cost_bps=CONFIG["cost_bps"],
)

spy_returns = spy["close"].pct_change().reindex(bt_results.index).fillna(0)
metrics = summary_metrics(bt_results["portfolio_return"])
display(pd.Series(metrics).to_frame("strategy"))

window_summary = summarize_oos_windows(bt_results, spy_returns)
display(window_summary)

plot_equity_curve(bt_results, spy_returns, title="Long-Only Active vs SPY")
plt.show()


## 7. 第四层：Regime Explanation

regime 在这份 notebook 里的默认角色是“解释层”。

只有当前三层都清楚后，才继续问：
- 高波动 / 高离散度是否真的更适合这类信号
- 这个关系在严格 OOS 下是否稳定
- 它能不能形成一个简单而稳健的过滤规则

不要把基于 rolling window 的相关性，直接升级成交易规则。


In [ ]:
# TODO:
# 1. 把市场状态变量放在 OOS 日期上做 ex-post 解释
# 2. 先看分桶稳定性，再决定是否尝试 regime filter
# 3. 若做 filter，必须与无 filter 版本进行严格 OOS 比较


## 8. 研究记录模板

| 层级 | 当前结论 | 是否站住 | 下一步 |
|---|---|---|---|
| Signal existence | 待更新 | 待判断 | 补 fold 稳定性与单调性 |
| Model comparison | 待更新 | 待判断 | 加入简单 baseline |
| Portfolio translation | 待更新 | 待判断 | 增加诊断型组合 |
| Regime explanation | 待更新 | 待判断 | 只做解释，不抢跑成规则 |

这份 notebook 的目标不是一次性写完，而是保证每一步结论都能回答一个清楚的问题。
